# **Scaling up RAG with Vector Databases**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Chunking Text for Retrieval-Augmented Generation Systems](#Chunking-Text-for-Retrieval-Augmented-Generation-Systems)
  - [Introduction](#Introduction)
  - [Understanding Document Chunking](#Understanding-Document-Chunking)
  - [Splitting Large Text into Chunks](#Splitting-Large-Text-into-Chunks)
  - [Inspecting the Input Data](#Inspecting-the-Input-Data)
  - [Adding Meaningful Metadata](#Adding-Meaningful-Metadata)
  - [Practical Usage Example](#Practical-Usage-Example)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
  - [Preserve Sentence Boundaries in Chunks](#Preserve-Sentence-Boundaries-in-Chunks)
  - [Enhance Text Chunking Skills](#Enhance-Text-Chunking-Skills)
  - [Organize Chunks with Metadata](#Organize-Chunks-with-Metadata)
  - [Enhance Text with Keyword Detection](#Enhance-Text-with-Keyword-Detection)
- [Storing and Managing Text Chunks in Vector Databases](#Storing-and-Managing-Text-Chunks-in-Vector-Databases)
  - [Introduction](#Introduction)
  - [Understanding Vector Databases](#Understanding-Vector-Databases)
  - [Why We Need Vector Databases for RAG](#Why-We-Need-Vector-Databases-for-RAG)
  - [Setting Up ChromaDB and Basic Configuration](#Setting-Up-ChromaDB-and-Basic-Configuration)
  - [Preparing Data and Adding Chunks to ChromaDB](#Preparing-Data-and-Adding-Chunks-to-ChromaDB)
  - [Updating and Managing Documents](#Updating-and-Managing-Documents)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
  - [Loading and chunking text](#Loading-and-chunking-text)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [1]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.9.6
🔢 NumPy        2.0.2
🐼 Pandas       2.3.3
📊 Matplotlib   3.9.4
🔥 PyTorch      2.8.0
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Chunking Text for Retrieval-Augmented Generation Systems

## Introduction

Welcome to the first lesson of our course on "**Scaling Up RAG with Vector Databases**"! This is the third course in our series, building on the foundations laid in the previous courses. Course 1 introduced a basic RAG pipeline, providing a foundational understanding of how retrieval and generation can be combined. Course 2 focused on text representation, with a particular emphasis on embeddings.

In this course, we'll focus on scaling your Retrieval-Augmented Generation (RAG) system by building and querying a vector database. You'll learn to preprocess documents, store chunk embeddings in ChromaDB, retrieve relevant chunks using advanced techniques like compound metadata filters and weighting, and construct prompts that can handle multiple context chunks. Additionally, we'll cover managing updates to your collection and large-scale ingestion using batch strategies.

Our journey begins with *document chunking*, a crucial preprocessing step that enhances the efficiency and effectiveness of vector databases in RAG systems. By the end of this lesson, you'll be able to break down a lengthy document into discrete segments, each tagged with essential metadata, paving the way for robust retrieval and storage in vector databases.

## Understanding Document Chunking

When dealing with Retrieval-Augmented Generation, you typically feed chunks of text — along with certain metadata — into downstream components, like embedding models or vector search engines. This **document chunking** process is crucial for several reasons:

1. **Manageability**: Smaller segments are easier to process. Many models have a maximum context length, meaning they can only handle a certain number of tokens at a time. If the input text exceeds this limit, the model may become inefficient or unable to process the request, leading to errors or truncated outputs. Additionally, even if a model can technically handle larger contexts, performance may degrade, resulting in slower processing times and reduced accuracy.

2. **Context Preservation**: A well-sized chunk still retains enough local context to be meaningful. Chunks should be neither too large (leading to potential memory issues) nor too small (risking the loss of context).

3. **Enhanced Retrieval**: When text is split rationally, you can retrieve only the relevant segments instead of searching through entire documents — this shortens query times and boosts accuracy.

Think of a real-world example: If you have a large reference manual, you wouldn't read the whole thing to answer a single question. Instead, you'd look for the exact section (or chunk) that pertains to your query. The same principle carries into text retrieval on a computer.

## Splitting Large Text into Chunks

Here's a sample function showcasing how to slice a long piece of text into parts of a specified word count:

In [1]:
def chunk_text(text, chunk_size=10):
    """
    Splits the given text into smaller chunks, each containing
    up to 'chunk_size' words. Returns a list of these chunk strings.
    """
    words = text.split()  # Tokenize by splitting on whitespace
    # Construct chunks by stepping through the words list in increments of chunk_size
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [2]:
sample_text = "This is a sample text that we will use to test our chunking function. It contains multiple sentences to make it more interesting."

chunks = chunk_text(sample_text)
for chunk in chunks:
    print(chunk)

This is a sample text that we will use to
test our chunking function. It contains multiple sentences to make
it more interesting.


After tokenizing the text, the function uses a list comprehension to iterate in steps of `chunk_size`. Each step produces a concise string of words that can be processed more easily by language models. Note that this method is a simplified approach and may not handle punctuation or sentence boundaries effectively, causing issues with *context preservation*; for more advanced chunking, consider using NLP libraries like NLTK or spaCy that can respect sentence boundaries.

## Inspecting the Input Data

Before moving on, let's briefly take a closer look at the dataset we'll be working with for chunking. Below is an example of two documents in JSON format:

```json
[
  {
    "id": 1,
    "content": "Hello world! This is a sample document used for testing chunk_text function."
  },
  {
    "id": 2,
    "content": "Another sample document. This is used for verifying the chunking of text in multiple documents. It includes additional sentences to provide a more comprehensive test case. By having a longer document, we can better assess how the chunking function performs when dealing with more extensive content."
  }
]
```

Our dataset consists of an array of items (or a list of dictionaries, in Python's terms), where each item has an `id` field that identifies the document and a `content` field containing the main text. After applying a chunking approach to each document, our data will be represented as smaller segments of text that are easier to process downstream, as we'll see later in the lesson.

## Adding Meaningful Metadata

So far, we have a method for chunking text, but in RAG systems, we often need to know *where* each piece came from and *what* it represents. That's where metadata comes in. Below is an example function that demonstrates how to loop through a structured dataset of documents, chunk each one, and store metadata for later use:

In [3]:
def load_and_chunk_dataset(data, chunk_size=10):
    """
    Iterates over a structured dataset of documents, splits each into chunks,
    and associates metadata (doc_id and chunk_id) with every piece.
    """
    all_chunks = []
    for doc in data:
        doc_id = doc["id"]
        doc_text = doc["content"]

        # Create smaller text segments from the original document
        doc_chunks = chunk_text(doc_text, chunk_size)

        # Label each chunk with its source identifier
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "text": chunk_str
                }
            )

    return all_chunks

Here's how it works step-by-step:

1. We iterate over `data`, which is a collection of different text entries in the dataset.
2. For each entry, the function extracts the `id` and `content` fields to process.
3. We apply the earlier `chunk_text` function to split the content into multiple pieces.
4. We then store relevant information — `doc_id` and `chunk_id` — along with the chunked text. This means every piece of text is *traceable* back to its origin, which can be crucial for retrieval and filtering in a vector database.

## Practical Usage Example

In this lesson, we'll use two sample documents to demonstrate the chunking process and to illustrate the concepts discussed. In the practice section, we'll be using a more realistic dataset for hands-on exercises; it can be found at `data/corpus.json`.

In [4]:
import json

# Open data/corpus.json file and load data
with open('data/corpus.json', 'r') as file:
    data = json.load(file)

chunked_docs = load_and_chunk_dataset(data, chunk_size=10)
print(f'Loaded and chunked {len(chunked_docs)} chunks from dataset.')
for doc in chunked_docs:
    print(doc)

Loaded and chunked 767 chunks from dataset.
{'doc_id': 1, 'chunk_id': 0, 'text': 'Artificial intelligence is transforming the way we approach complex problems'}
{'doc_id': 1, 'chunk_id': 1, 'text': 'in computing. Recent breakthroughs in machine learning have enabled faster'}
{'doc_id': 1, 'chunk_id': 2, 'text': 'data processing and smarter algorithms. The future of technology is'}
{'doc_id': 1, 'chunk_id': 3, 'text': 'expected to integrate AI into every facet of life.'}
{'doc_id': 2, 'chunk_id': 0, 'text': 'Robotics continues to advance with improved sensors and autonomous capabilities.'}
{'doc_id': 2, 'chunk_id': 1, 'text': 'Industrial applications are increasingly adopting these technologies for enhanced efficiency.'}
{'doc_id': 2, 'chunk_id': 2, 'text': 'Engineers remain optimistic about the seamless integration of robots in'}
{'doc_id': 2, 'chunk_id': 3, 'text': 'everyday manufacturing.'}
{'doc_id': 3, 'chunk_id': 0, 'text': 'Quantum computing has made significant strides in recent

This output clearly illustrates both the benefits and limitations of our simple chunking approach. While we've successfully broken down the documents into smaller pieces with proper metadata tracking, the quality of some chunks is problematic. For example, chunk 1 of document 1 contains only the fragment `'chunk_text function.'`, which lacks sufficient context to be meaningful on its own. Similarly, other chunks cut across sentence boundaries, creating segments that might be difficult for retrieval systems to properly interpret.

These issues highlight why more sophisticated chunking strategies that respect semantic boundaries (like sentences or paragraphs) are essential for **context preservation** in production RAG systems. Poor chunking can significantly reduce the effectiveness of retrieval, as chunks without adequate context may not match relevant queries or might provide incomplete information. In the practice section, we'll explore more advanced techniques that better preserve context and create more meaningful chunks.

## Conclusion and Next Steps

In this lesson, we focused on *why* and *how* to chunk larger text documents. We learned that chunking:

* Makes text blocks suitably sized for processing with language models.
* Preserves local context by grouping words carefully.
* Allows for the attachment of metadata that makes future retrieval more powerful.

Next, we'll embed these chunks and store them in a vector database so we can efficiently search within them. Feel free to **practice** splitting your own documents and confirm the results before moving on to the upcoming exercises. By mastering the chunking and metadata process, you're well on your way to creating scalable, retrieval-enhanced applications.

## Preserve Sentence Boundaries in Chunks

You've learned how to split text into chunks based on word count. Now, let's enhance your skills by refining the `chunk_text` function to better preserve sentence boundaries, which will help maintain the context and coherence of the text segments.

To achieve this, follow these steps:

* Modify the `chunk_text` function to split the text not only by word count but also by punctuation marks such as periods, exclamation points, and question marks. This ensures that sentences remain intact within chunks. *Hint*: you can use `re.split('[.!?]+', text)` to split `text` on multiple characters.

* After splitting by punctuation, group the resulting sentences into chunks. Add whole sentences to the current chunk while the total stays at or below `chunk_size` words; when the next sentence would push the count over `chunk_size`, close the current chunk and start a new one with that sentence. Never split a sentence across two chunks.

* If a single sentence has more than `chunk_size` words, keep it intact as its own chunk even though that chunk exceeds `chunk_size`: sentence boundaries take precedence over the word limit. With the default `chunk_size=30`, two documents in `corpus.json` start with a 31-word sentence, so a correct implementation prints exactly two chunks of 31 words and none longer.

Once you've implemented these changes, test your function with a sample text and observe how the chunks differ. Notice how preserving sentence boundaries can enhance the readability and context of each chunk. This exercise will deepen your understanding of text chunking and prepare you for more advanced techniques. Enjoy the challenge and happy coding!

For this and the next practices, we'll be using a dataset located at `data/corpus.json`. This dataset contains a collection of documents, each with fields such as `id`, `title`, `content`, `category`, `tags`, and `date`. Here's a sample entry from the file:

```json
{
  "id": 1,
  "title": "Revolutionizing Computing with AI",
  "content": "Artificial intelligence is transforming the way we approach complex problems in computing. Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms. The future of technology is expected to integrate AI into every facet of life.",
  "category": "Technology",
  "tags": ["AI", "machine learning", "computing", "innovation"],
  "date": "2025-02-01"
}
```

In [6]:
import os
import json
import re

def chunk_text(text, chunk_size=30):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of chunk strings.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    sentences = re.split('[.?!]+', text)

    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed
    chunks = []
    chunk_in_progress = False
    chunk_words = []
    chunk_word_count = 0
    for i, sentence in enumerate(sentences):
        if sentence:  # Only consider non-empty sentences in order to create only non-empty chunks
            words = sentence.split()
            # chunk_words += words
            # chunk_word_count += len(words)
            if chunk_word_count + len(words) >= chunk_size and not chunk_in_progress:  # keep sentence intact as its own chunk
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) < chunk_size and not chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) < chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) == chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) > chunk_size and chunk_in_progress:
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True

    if len(chunk_words):
        chunks.append(" ".join(chunk_words))

    return chunks


def load_and_chunk_dataset(file_path, chunk_size=30):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)
    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        doc_category = doc.get("category", "general")
        doc_chunks = chunk_text(doc_text, chunk_size)
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "category": doc_category,
                "text": chunk_str
            })
    return all_chunks

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")
chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=30)
print("Loaded and chunked", len(chunked_docs), "chunks from dataset.")
for c in chunked_docs:
    print(c)

Loaded and chunked 353 chunks from dataset.
{'doc_id': 0, 'chunk_id': 0, 'category': 'Technology', 'text': 'Artificial intelligence is transforming the way we approach complex problems in computing Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms'}
{'doc_id': 0, 'chunk_id': 1, 'category': 'Technology', 'text': 'The future of technology is expected to integrate AI into every facet of life'}
{'doc_id': 1, 'chunk_id': 0, 'category': 'Technology', 'text': 'Robotics continues to advance with improved sensors and autonomous capabilities Industrial applications are increasingly adopting these technologies for enhanced efficiency'}
{'doc_id': 1, 'chunk_id': 1, 'category': 'Technology', 'text': 'Engineers remain optimistic about the seamless integration of robots in everyday manufacturing'}
{'doc_id': 2, 'chunk_id': 0, 'category': 'Technology', 'text': 'Quantum computing has made significant strides in recent years, pushing the boundaries of wh

## Enhance Text Chunking Skills

Well done on refining your chunking skills by preserving sentence boundaries in the previous exercise. Now, let's elevate your skills by introducing a user-configurable **overlap** between adjacent chunks. This enhancement will help maintain the flow of context across chunks, especially in longer texts. Your mission is to enhance the `chunk_text` function to allow for this overlap.

Specifically, you should modify the `chunk_text` function to include an overlap feature. This involves reusing a certain number of words from the end of one chunk as the beginning of the next. The overlap should be user-configurable, allowing you to specify how many words to reuse.

Once you've implemented the overlap, test your function with sample text and observe how the chunks transition more smoothly. For example, if you have a text with 50 words and a `chunk_size` of 10 with an overlap of 2, the first chunk might contain words 1-10, and the second chunk would start with words 9-18, and so on. This overlap ensures that important context is not lost between chunks.

Enjoy the challenge, and happy coding!

In [1]:
import os
import json

def chunk_text(text, chunk_size=10, overlap=2):
    """
    Splits the given text into chunks of size 'chunk_size' with a specified overlap.
    Returns a list of chunk strings.
    
    Args:
        text (str): The input text to chunk
        chunk_size (int): The size of each chunk in words
        overlap (int): Number of words to overlap between chunks
    """
    # TODO: Split the text into words
    words = text.split()
    
    # TODO: Calculate the step size
    step_size = chunk_size - overlap
    
    # TODO: Create chunks with overlap
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), step_size)]
    
    return chunks

def load_and_chunk_dataset(file_path, chunk_size=30, overlap=5):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)
    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        doc_category = doc.get("category", "general")
        doc_chunks = chunk_text(doc_text, chunk_size, overlap)
        
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "category": doc_category,
                "text": chunk_str
            })
    return all_chunks

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")

# Test the overlapping chunks
chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=10, overlap=2)
print("Loaded and chunked", len(chunked_docs), "chunks from dataset.")

# Print chunks to see the overlap in action
for i, chunk in enumerate(chunked_docs):
    print(f"\nChunk {i}:")
    print(chunk["text"])

Loaded and chunked 937 chunks from dataset.

Chunk 0:
Artificial intelligence is transforming the way we approach complex problems

Chunk 1:
complex problems in computing. Recent breakthroughs in machine learning have

Chunk 2:
learning have enabled faster data processing and smarter algorithms. The

Chunk 3:
algorithms. The future of technology is expected to integrate AI

Chunk 4:
integrate AI into every facet of life.

Chunk 5:
Robotics continues to advance with improved sensors and autonomous capabilities.

Chunk 6:
autonomous capabilities. Industrial applications are increasingly adopting these technologies for

Chunk 7:
technologies for enhanced efficiency. Engineers remain optimistic about the seamless

Chunk 8:
the seamless integration of robots in everyday manufacturing.

Chunk 9:
Quantum computing has made significant strides in recent years, pushing

Chunk 10:
years, pushing the boundaries of what computers can achieve. Researchers

Chunk 11:
achieve. Researchers are explori

## Organize Chunks with Metadata

Well done! Now, let's elevate your script by adding a new layer of organization to your chunks. Your mission is to extend your script to incorporate a **category** field from each document as metadata for every text chunk. This will help you organize and retrieve relevant chunks more effectively.

Here's what you need to do:

* Examine each document for a **category** field. If a document lacks a category, default it to something like `"general"` or `"misc"`.
* Attach this category as metadata to each chunk, along with `doc_id` and `chunk_id`.

After processing, print each chunk with its `doc_id`, `chunk_id`, and `category` to observe how these metadata fields enhance the organization of your data.

For instance, if you have a document with the category `"Technology"`, and it gets split into three chunks, each chunk should carry this category information. This exercise will deepen your understanding of metadata management and prepare you for more advanced data retrieval techniques. Embrace the challenge and see how far you can go!

In [2]:
import os
import json
import re

def chunk_text(text, chunk_size=30):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of chunk strings.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    sentences = re.split('[.?!]+', text)

    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed
    chunks = []
    chunk_in_progress = False
    chunk_words = []
    chunk_word_count = 0
    for i, sentence in enumerate(sentences):
        if sentence:  # Only consider non-empty sentences in order to create only non-empty chunks
            words = sentence.split()
            # chunk_words += words
            # chunk_word_count += len(words)
            if chunk_word_count + len(words) >= chunk_size and not chunk_in_progress:  # keep sentence intact as its own chunk
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) < chunk_size and not chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) < chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) == chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) > chunk_size and chunk_in_progress:
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True

    if len(chunk_words):
        chunks.append(" ".join(chunk_words))

    return chunks

def load_and_chunk_dataset(file_path, chunk_size=30):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)

    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        # TODO: Get the category from the document, use "general" as default if not found
        doc_category = doc.get("category", "general")

        # TODO: Split the document text into chunks using the chunk_text function
        doc_chunks = chunk_text(doc_text, chunk_size)

        # TODO: Iterate over the chunks and create a dictionary for each chunk with doc_id, chunk_id, category, and text
        # TODO: Append each chunk dictionary to all_chunks
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "category": doc_category,
                    "text": chunk_str
                }
            )

    return all_chunks

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")
chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=30)
print(f'Loaded and chunked {len(chunked_docs)} chunks from dataset.')
for chunk in chunked_docs:
    print(chunk)

Loaded and chunked 353 chunks from dataset.
{'doc_id': 0, 'chunk_id': 0, 'category': 'Technology', 'text': 'Artificial intelligence is transforming the way we approach complex problems in computing Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms'}
{'doc_id': 0, 'chunk_id': 1, 'category': 'Technology', 'text': 'The future of technology is expected to integrate AI into every facet of life'}
{'doc_id': 1, 'chunk_id': 0, 'category': 'Technology', 'text': 'Robotics continues to advance with improved sensors and autonomous capabilities Industrial applications are increasingly adopting these technologies for enhanced efficiency'}
{'doc_id': 1, 'chunk_id': 1, 'category': 'Technology', 'text': 'Engineers remain optimistic about the seamless integration of robots in everyday manufacturing'}
{'doc_id': 2, 'chunk_id': 0, 'category': 'Technology', 'text': 'Quantum computing has made significant strides in recent years, pushing the boundaries of wh

## Enhance Text with Keyword Detection

Good job, you got to the final exercise in this unit! Your last goal here is to enhance the `chunk_text` function to detect and tag specific **keywords** within each chunk, enriching the metadata and making your chunks not only well-structured but also contextually insightful.

Begin by defining a small set of **keywords** you wish to track within your text chunks. These could be words like `"Artificial Intelligence"`, `"technology"`, or `"cinema"`. Next, modify the `chunk_text` function to scan each chunk for these keywords. If a keyword is found, ensure it is tagged in the metadata associated with that chunk.

Run your enhanced chunking function over a short text or JSON dataset, such as the one provided in `data/corpus.json`. Print out both the chunk and its metadata to verify that any encountered keywords were correctly identified and tagged.

For instance, if a chunk contains the sentence `"This is a sample document used for testing the chunk_text function,"` and your keywords include `"testing"` and `"chunking"`, the metadata should reflect these keywords. This exercise will deepen your understanding of text processing and metadata management, setting the stage for more advanced data retrieval techniques. Dive in and embrace the challenge!

In [4]:
import os
import json
import re

def chunk_text(text, chunk_size=30, keywords_to_track=["Artificial Intelligence", "technology", "cinema"]):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of dictionaries containing chunk text and found keywords.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    sentences = re.split('[.?!]+', text)

    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed
    chunks = []
    chunk_in_progress = False
    chunk_words = []
    chunk_word_count = 0

    def create_chunk(chunk_words, keywords_to_track):
        chunk_text = " ".join(chunk_words)
        # TODO: Create a set to store found keywords in this chunk
        found_keywords = set()
        # TODO: Check each word in the chunk against keywords
        # If a word is in keywords, add it to found keywords
        for keyword in keywords_to_track:
            if keyword.lower() in chunk_text.lower():
                found_keywords.add(keyword)

        found_keywords = sorted(list(found_keywords))

        chunk = {
            "text": chunk_text,
            # TODO: Add the found keywords to the chunk metadata
            "keywords": found_keywords
        }
        return chunk
        
    for i, sentence in enumerate(sentences):
        if sentence:  # Only consider non-empty sentences in order to create only non-empty chunks
            words = sentence.split()
            # chunk_words += words
            # chunk_word_count += len(words)
            if chunk_word_count + len(words) >= chunk_size and not chunk_in_progress:  # keep sentence intact as its own chunk
                chunk_words += words
                chunk_word_count += len(words)

                chunk = create_chunk(chunk_words, keywords_to_track)
                chunks.append(chunk)
                
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) < chunk_size and not chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) < chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) == chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)

                chunk = create_chunk(chunk_words, keywords_to_track)
                chunks.append(chunk)
                
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) > chunk_size and chunk_in_progress:
                chunk = create_chunk(chunk_words, keywords_to_track)
                chunks.append(chunk)
                
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True

    if len(chunk_words):
        chunk = create_chunk(chunk_words, keywords_to_track)
        chunks.append(chunk)

    return chunks

def load_and_chunk_dataset(file_path, chunk_size=30, keywords_to_track=["Artificial Intelligence", "technology", "cinema"]):
    """
    Loads a dataset from JSON 'file_path', splits each document into smaller chunks,
    and detects keywords in each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)

    all_chunks = []
    for doc_id, doc in enumerate(data):
        doc_text = doc["content"]
        doc_category = doc.get("category", "general")
        doc_chunks = chunk_text(doc_text, chunk_size, keywords_to_track)

        for chunk_id, chunk_data in enumerate(doc_chunks):
            # Add doc_id, chunk_id and category to chunk_data
            chunk_data.update(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "category": doc_category
                }
            )
            all_chunks.append(chunk_data)

    return all_chunks

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")
# Define keywords to track
keywords_to_track = ["Artificial Intelligence", "technology", "cinema"]
# Load and chunk the dataset with keyword detection
chunked_docs = load_and_chunk_dataset(dataset_file, chunk_size=30, keywords_to_track=keywords_to_track)

print(f'Loaded and chunked {len(chunked_docs)} chunks from dataset.')
for chunk in chunked_docs:
    print('\nChunk:')
    print(f'Text: {chunk["text"]}')
    print(f'Keywords found: {chunk["keywords"]}')

Loaded and chunked 353 chunks from dataset.

Chunk:
Text: Artificial intelligence is transforming the way we approach complex problems in computing Recent breakthroughs in machine learning have enabled faster data processing and smarter algorithms
Keywords found: ['Artificial Intelligence']

Chunk:
Text: The future of technology is expected to integrate AI into every facet of life
Keywords found: ['technology']

Chunk:
Text: Robotics continues to advance with improved sensors and autonomous capabilities Industrial applications are increasingly adopting these technologies for enhanced efficiency
Keywords found: []

Chunk:
Text: Engineers remain optimistic about the seamless integration of robots in everyday manufacturing
Keywords found: []

Chunk:
Text: Quantum computing has made significant strides in recent years, pushing the boundaries of what computers can achieve
Keywords found: []

Chunk:
Text: Researchers are exploring novel AI-powered approaches to solving problems once consider

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Storing and Managing Text Chunks in Vector Databases

## Introduction

Hello there, welcome to the second lesson of our "**Scaling Up RAG with Vector Databases**" course! In the previous unit, you explored how to break large documents into smaller *chunks* and attach useful metadata (like `doc_id`, `chunk_id`, and labels such as `category`). These chunks are essential for structuring data in a way that makes retrieval easier. In this lesson, we'll build on that groundwork by showing you how to store them in a **vector database**. One popular choice is **ChromaDB** — a specialized, open-source database designed for high-speed, *semantic* querying of vectors. By switching from keyword-based searches to semantic searches, your RAG system will retrieve relevant information more efficiently. Let's dive in!

## Understanding Vector Databases

A vector database stores data in the form of numerical *vectors* that capture the semantic essence of texts (or other data). The database then uses similarity metrics—rather than literal word matches—so that conceptually similar items are stored close together. This means searches on vector databases can retrieve contextually relevant results even when keywords are absent. By leveraging *approximate* or *exact* nearest-neighbor strategies for similarity, vector databases can scale to handle millions or billions of vectors while still providing quick query responses. This makes them especially suitable for RAG systems, which rely on fast semantic lookups across large collections of text.

## Why We Need Vector Databases for RAG

Before we explore how to set up a vector database, let's look at why it's a crucial component of a RAG pipeline:

1. **Semantic Retrieval**: By embedding text into *vectors*, queries can match documents based on *meaning* rather than strict keyword matches. This yields more accurate and context-sensitive search results.
2. **Scalability**: Specialized vector databases handle large datasets efficiently, allowing you to store and query vast libraries of text chunks without sacrificing performance.
3. **Richer Context**: Embeddings capture nuanced relationships among chunks, ensuring that related information is surfaced even when it doesn't use the exact same terms.
4. **Easy Updates**: Vector databases (like ChromaDB) often allow you to add and remove chunks on the fly, so your collection stays in sync with new or evolving information.

## Setting Up ChromaDB and Basic Configuration

Now, let's jump into coding with **ChromaDB**, our chosen vector database. Here's how to set up a ChromaDB client:

In [2]:
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions

def build_chroma_collection(chunks):
    # Use a Sentence Transformer model for embeddings
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

    # Create a ChromaDB client with default settings
    client = Client(Settings())

    # Either get an existing collection or create a new one
    collection = client.get_or_create_collection(
        name='rag_collection',
        embedding_function=embed_func
    )
    # ... continues

**How It Works**:

* **Embedding Setup**: We define a `SentenceTransformerEmbeddingFunction` to generate vectors for the text chunks. The model we're using, `all-MiniLM-L6-v2`, is a lightweight but powerful sentence transformer that maps sentences to a 384-dimensional dense vector space. It's popular for RAG applications because it balances efficiency (small size, fast inference) with strong semantic understanding capabilities.

* **Client Configuration**: `Client(Settings())` connects to ChromaDB with default settings. By default, ChromaDB creates an in-memory store for quick experimentation. You can customize its behavior (for example, specifying a file path for persistence or enabling other features) by passing additional parameters to `Settings()`.

* **Collection Management**: `get_or_create_collection` checks if a collection named `'rag_collection'` exists; if not, it creates a new one. A collection in ChromaDB is a logical container that groups related documents and their embeddings together, similar to a table in a traditional database but optimized for vector similarity operations. Collections allow you to organize your vector data into separate namespaces, making it possible to maintain multiple distinct sets of documents with different embedding models or for different use cases.

## Preparing Data and Adding Chunks to ChromaDB

After setting up your client, embedding function, and creating the collection, the next step is to prepare your chunks for insertion and add them to the collection:

In [3]:
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions

def build_chroma_collection(chunks):
    # Use a Sentence Transformer model for embeddings
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

    # Create a ChromaDB client with default settings
    client = Client(Settings())

    # Either get an existing collection or create a new one
    collection = client.get_or_create_collection(
        name='rag_collection',
        embedding_function=embed_func
    )

    # Prepare the data: texts, IDs, and metadata
    texts = [chunk["content"] for chunk in chunks]
    ids = [f"chunk_{chunk['doc_id']}_{chunk['chunk_id']}" for chunk in chunks]
    metadatas = [
        {
            'doc_id': chunk['doc_id'],
            'chunk_id': chunk['chunk_id'],
            'category': chunk['category']
        } for chunk in chunks
    ]

    # Add the documents (chunks) to the collection
    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    return collection

**Key Points**:

* **Data Grouping**: Each chunk is mapped to its text, a unique ID, and metadata. These are used during retrieval and future reference.

* **Seamless Insertion**: Calling `collection.add()` handles embedding and stores everything for quick semantic searches.

## Updating and Managing Documents

ChromaDB allows you to keep your collection up to date with new or modified information. Below is an example of adding and then deleting a "document" (or chunk) after the collection has already been created:

In [5]:
# Example chunks to showcase adding them to a new collection
example_chunks = [
    {"doc_id": 0, "chunk_id": 0, "category": "ai", "content": "RAG stands for Retrieval-Augmented Generation."},
    {"doc_id": 0, "chunk_id": 1, "category": "ai", "content": "A crucial component of a RAG pipeline is the Vector Database."},
    {"doc_id": 1, "chunk_id": 0, "category": "finance", "content": "Accurate data is essential in finance."},
]
collection = build_chroma_collection(example_chunks)

# Prepare a new chunk to add
new_document = {
    "doc_id": 2,
    "chunk_id": 0,
    "category": "food",
    "content": "Bananas are yellow fruits rich in potassium."
}

# Construct a unique ID for the new document
# Format: "chunk_{doc_id}_{chunk_id}" (e.g., "chunk_2_0")
doc_id = f"chunk_{new_document['doc_id']}_{new_document['chunk_id']}"

# Add the new chunk to the existing collection
collection.add(
    documents=[new_document["content"]],  # The text content to be embedded
    metadatas=[                           # Metadata for filtering and context
        {
            "doc_id": new_document["doc_id"],
            "chunk_id": new_document["chunk_id"],
            "category": new_document["category"]
        }
    ],
    ids=[doc_id]                          # Unique identifier for this chunk
)

# If needed, remove the chunk by its unique ID
# For example, if the information about bananas becomes outdated
# collection.delete(ids=[doc_id])  # Using the same ID: "chunk_2_0"

**Key Points**:

* `example_chunks` is our initial list of text chunks with their metadata, while `new_document` is a new chunk we want to add to our existing collection.
* `doc_id` is a unique identifier string created by combining document ID and chunk ID (e.g., `"chunk_2_0"`).
* **Why Unique IDs Matter**: Each chunk needs a unique identifier so ChromaDB can reference it later for updates, deletions, or retrieval. By combining `doc_id` and `chunk_id` into a string like `"chunk_2_0"`, we ensure each chunk has a distinct ID while maintaining its relationship to the source document.
* **Adding**: Calling `collection.add()` again embeds the new document automatically, so your collection grows in real time.
* **Deleting**: When old or incorrect information becomes obsolete—such as a product description that's been updated or regulatory data that's changed—you can simply remove it by its ID. In real-world scenarios, data about policies, facts, or even entire documents can become outdated over time, and selectively deleting chunks lets you keep your database accurate without a full rebuild.

This flexibility means you can keep your RAG environment updated without fully restructuring your database.

## Conclusion and Next Steps

By storing text chunks in a **vector database**, you've laid the *foundation* for faster, more *semantically aware* retrieval. You know how to create, update, and manage a ChromaDB collection—crucial skills for any large-scale RAG system.

In the next lesson, you'll learn how to **query** the vector database to fetch the most relevant chunks and feed them into a language model. That's where the real magic of producing context-rich, accurate responses shines! For now, feel free to explore different embedding models or try adding and deleting a variety of chunks. When you're ready, proceed to the practice exercises to cement these concepts and further refine your RAG workflow.

## Loading and chunking text

Now, let's put your newfound knowledge into practice by exploring the fascinating world of **vector databases** with **ChromaDB**.

Your mission is to load a small JSON file containing a few short documents, split each document into smaller, manageable chunks, and then build a ChromaDB collection from these chunks. Here's what you need to do in the `load_and_chunk_dataset` function:

* Load the dataset from the JSON file located at `data/corpus.json`. Each document in this file has the keys: `id`, `title`, `content`, and `category`.
* Extract the `category` and `id` from each document.
* Use the provided `chunk_text` function to split the content of each document into smaller segments.
* Create a dictionary for each chunk that includes metadata such as `doc_id`, `chunk_id`, `category`, and the chunk `text`.

By the end of this exercise, you'll have a solid understanding of how to prepare and store text chunks in a vector database, setting the stage for efficient semantic retrieval. Enjoy the process and happy coding!

In [6]:
import os
import json
import re
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils import embedding_functions

def chunk_text(text, chunk_size=30):
    """
    Splits the given text into chunks of size 'chunk_size', preserving sentence boundaries.
    Returns a list of chunk strings.
    """
    # TODO: Split text into sentences using regex
    # Hint: Use re.split() with appropriate punctuation marks
    sentences = re.split('[.?!]+', text)

    # TODO: Process sentences into chunks while respecting chunk_size
    # Hint: Keep track of word count and create new chunks when needed
    chunks = []
    chunk_in_progress = False
    chunk_words = []
    chunk_word_count = 0
    for i, sentence in enumerate(sentences):
        if sentence:  # Only consider non-empty sentences in order to create only non-empty chunks
            words = sentence.split()
            # chunk_words += words
            # chunk_word_count += len(words)
            if chunk_word_count + len(words) >= chunk_size and not chunk_in_progress:  # keep sentence intact as its own chunk
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) < chunk_size and not chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) < chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True
            elif chunk_word_count + len(words) == chunk_size and chunk_in_progress:
                chunk_words += words
                chunk_word_count += len(words)
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
            elif chunk_word_count + len(words) > chunk_size and chunk_in_progress:
                chunks.append(" ".join(chunk_words))
                chunk_in_progress = False
                chunk_words = []
                chunk_word_count = 0
                chunk_words += words
                chunk_word_count += len(words)
                chunk_in_progress = True

    if len(chunk_words):
        chunks.append(" ".join(chunk_words))

    return chunks

def load_and_chunk_dataset(file_path, chunk_size=30):
    """
    Loads a dataset from JSON 'file_path', then splits each document into smaller chunks.
    Metadata such as 'doc_id' and 'category' is included with each chunk.
    """
    with open(file_path, "r") as f:
        data = json.load(f)

    all_chunks = []
    for doc in data:
        doc_text = doc["content"]
        doc_id = doc["id"]
        doc_category = doc.get("category", "general")
        
        doc_chunks = chunk_text(doc_text, chunk_size)
        for chunk_id, chunk_str in enumerate(doc_chunks):
            all_chunks.append(
                {
                    "doc_id": doc_id,
                    "chunk_id": chunk_id,
                    "category": doc_category,
                    "text": chunk_str
                }
            )

    return all_chunks

def build_chroma_collection(chunks, collection_name="rag_collection"):
    """
    Builds or retrieves a ChromaDB collection, embedding each chunk using a SentenceTransformer.
    Adds all chunks in the 'chunks' list to the collection for fast retrieval.
    """
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=model_name)

    client = Client(Settings())
    collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=embed_func
    )

    texts = [c["text"] for c in chunks]
    ids = [f"chunk_{c['doc_id']}_{c['chunk_id']}" for c in chunks]
    metadatas = [
        {"doc_id": c["doc_id"], "chunk_id": c["chunk_id"], "category": c["category"]}
        for c in chunks
    ]

    collection.add(documents=texts, metadatas=metadatas, ids=ids)
    return collection

current_dir = os.getcwd()
dataset_file = os.path.join(current_dir, "data", "corpus.json")

chunked_docs = load_and_chunk_dataset(dataset_file)
collection = build_chroma_collection(chunked_docs)

total_docs = collection.count()
print(f'ChromaDB collection created with {total_docs} documents.')

ChromaDB collection created with 355 documents.


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[CodeSignal: Scaling up RAG with Vector Databases Course](https://codesignal.com/learn/courses/scaling-up-rag-with-vector-databases)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)